# 深層強化学習

表形式 Q 学習は、状態と行動の組み合わせをすべて表に持てる環境で強力に働きます。観測が画像、連続値、長い履歴になると、表はすぐ破綻します。深層強化学習は、Q 値や方策をニューラルネットで近似し、似た状態へ経験を共有する方法です。

DQN は代表的な入口です。状態特徴を入力し、各行動の Q 値を出し、ベルマン target に近づくよう重みを更新します。難しさは、教師信号も同じ学習中のネットワークから作られる点にあります。

表をニューラルネットに置き換えると、似た状態へ経験を共有できる一方で、更新が不安定になります。replay buffer と target network は飾りではなく、偏った経験と動き続ける教師信号を抑えるための仕組みです。

## 表から関数近似へ

表形式では Q[s][a] を直接更新します。DQN では q_network(features(s))[a] を更新します。式は似ていますが、1 回の更新が共有重みを通じて他の状態にも影響します。

In [ ]:
import math
import random

random.seed(0)

gamma = 0.92
learning_rate = 0.04
actions = ['left', 'right']

# 0 が左端、4 が右端。右端へ到達すると報酬 1 で終了する小さな環境。
transitions = [
    (0, 1, 0.0, 1, False),
    (1, 1, 0.0, 2, False),
    (2, 1, 0.0, 3, False),
    (3, 1, 1.0, 4, True),
    (1, 0, 0.0, 0, False),
    (2, 0, 0.0, 1, False),
    (3, 0, 0.0, 2, False),
]

def features(state):
    position = state / 4
    return [1.0, position, position * position]

def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

def relu(values):
    return [max(0.0, v) for v in values]

def relu_grad(values):
    return [1.0 if v > 0 else 0.0 for v in values]

def rounded(values, ndigits=3):
    return [round(v, ndigits) for v in values]

## 小さな Q ネットワーク

入力は状態特徴、出力は行動ごとの Q 値です。隠れ層を 1 つだけ持つ MLP でも、表とは違って状態間の滑らかな関係を表せます。

In [ ]:
network = {
    'w1': [
        [0.10, -0.20, 0.05],
        [0.00, 0.15, 0.20],
        [-0.10, 0.10, 0.25],
        [0.20, 0.05, -0.10],
    ],
    'b1': [0.0, 0.0, 0.0, 0.0],
    'w2': [
        [0.10, -0.05, 0.08, 0.02],
        [-0.04, 0.06, 0.01, 0.09],
    ],
    'b2': [0.0, 0.0],
}

def forward(params, state):
    x = features(state)
    z1 = [dot(row, x) + b for row, b in zip(params['w1'], params['b1'])]
    h = relu(z1)
    q = [dot(row, h) + b for row, b in zip(params['w2'], params['b2'])]
    return {'x': x, 'z1': z1, 'h': h, 'q': q}

def q_values(params, state):
    return forward(params, state)['q']

for state in range(5):
    print(state, rounded(q_values(network, state)))

## TD target で 1 遷移を更新する

DQN の損失は、選んだ行動の Q 値だけを target に近づけます。終端遷移では次状態の価値を足しません。

In [ ]:
def copy_params(params):
    copied = {}
    for key, value in params.items():
        copied[key] = [row[:] if isinstance(row, list) else row for row in value]
    return copied

def td_target(params, reward, next_state, done):
    if done:
        return reward
    return reward + gamma * max(q_values(params, next_state))

def apply_update(params, transition, target_params=None, lr=learning_rate, clip=1.0):
    state, action, reward, next_state, done = transition
    target_source = target_params if target_params is not None else params
    cache = forward(params, state)
    pred = cache['q'][action]
    target = td_target(target_source, reward, next_state, done)
    error = pred - target
    error = max(-clip, min(clip, error))

    grad_q = [0.0 for _ in actions]
    grad_q[action] = error

    grad_w2 = [[grad_q[i] * h for h in cache['h']] for i in range(len(actions))]
    grad_b2 = grad_q[:]

    grad_h = []
    for j in range(len(cache['h'])):
        grad_h.append(sum(params['w2'][i][j] * grad_q[i] for i in range(len(actions))))
    grad_z1 = [g * r for g, r in zip(grad_h, relu_grad(cache['z1']))]
    grad_w1 = [[grad_z1[i] * x for x in cache['x']] for i in range(len(cache['z1']))]
    grad_b1 = grad_z1

    for i in range(len(params['w2'])):
        for j in range(len(params['w2'][i])):
            params['w2'][i][j] -= lr * grad_w2[i][j]
        params['b2'][i] -= lr * grad_b2[i]
    for i in range(len(params['w1'])):
        for j in range(len(params['w1'][i])):
            params['w1'][i][j] -= lr * grad_w1[i][j]
        params['b1'][i] -= lr * grad_b1[i]

    return {'prediction': pred, 'target': target, 'error': error}

probe = apply_update(network, transitions[3], lr=0.01)
print({key: round(value, 3) for key, value in probe.items()})
print('state 3 Q:', rounded(q_values(network, 3)))

## online 更新の不安定さ

同じネットワークが予測と target 作成の両方に使われると、学習先が毎回動きます。さらに連続した遷移だけを使うと、似た経験に偏ります。DQN はこの 2 つに replay buffer と target network で対処します。

In [ ]:
online = copy_params(network)
log = []
for step in range(500):
    transition = transitions[step % len(transitions)]
    result = apply_update(online, transition, lr=0.035)
    if step in [0, 1, 2, 50, 200, 499]:
        log.append((step, transition[:3], result['target'], result['prediction']))

for row in log:
    step, transition_head, target, prediction = row
    print(f'step={step:3d} transition={transition_head} target={target:.3f} prediction={prediction:.3f}')

for state in range(5):
    print('online', state, rounded(q_values(online, state)))

## replay buffer は経験を混ぜ直す

Replay buffer は過去の遷移をため、ランダムに mini-batch を作ります。時間的に隣り合う経験だけで更新し続ける偏りを弱め、同じ経験を複数回使えるようにします。

In [ ]:
buffer = []
for episode in range(60):
    for transition in transitions:
        buffer.append(transition)
        if len(buffer) > 80:
            buffer.pop(0)

def sample_batch(buffer, batch_size, seed):
    rng = random.Random(seed)
    return [buffer[i] for i in rng.sample(range(len(buffer)), batch_size)]

batch = sample_batch(buffer, 5, seed=3)
for item in batch:
    print(item)

## target network は教師信号を固定する

Target network は online network のコピーです。数ステップごとにだけ同期し、TD target の揺れを抑えます。これにより、予測側は動いても教師側はしばらく固定されます。

In [ ]:
online = copy_params(network)
target = copy_params(online)
losses = []

for step in range(600):
    batch = sample_batch(buffer, 4, seed=step)
    batch_loss = 0.0
    for transition in batch:
        result = apply_update(online, transition, target_params=target, lr=0.03)
        batch_loss += result['error'] ** 2
    losses.append(batch_loss / len(batch))
    if (step + 1) % 100 == 0:
        target = copy_params(online)

print('first losses:', [round(v, 4) for v in losses[:5]])
print('last losses :', [round(v, 4) for v in losses[-5:]])
for state in range(5):
    print('targeted', state, rounded(q_values(online, state)))

## 一般化は利点でもリスクでもある

ネットワークは状態特徴を共有するため、ある状態の更新が別の状態の Q 値も動かします。近い状態なら学習効率が上がり、関係の薄い状態まで動くと過大評価や発散の原因になります。

In [ ]:
probe_net = copy_params(online)
before = {state: q_values(probe_net, state)[:] for state in range(5)}
apply_update(probe_net, (3, 1, 1.0, 4, True), target_params=target, lr=0.08)
after = {state: q_values(probe_net, state)[:] for state in range(5)}

for state in range(5):
    delta = [after[state][a] - before[state][a] for a in range(len(actions))]
    print(state, 'delta', rounded(delta, 4))

## Double DQN の考え方

DQN は max_a Q(next_state, a) を target に使うため、誤差を含む推定値の最大を選び、過大評価が起きやすくなります。Double DQN は行動選択を online network、値評価を target network に分けます。

In [ ]:
def dqn_target(target_params, reward, next_state, done):
    if done:
        return reward
    return reward + gamma * max(q_values(target_params, next_state))

def double_dqn_target(online_params, target_params, reward, next_state, done):
    if done:
        return reward
    online_q = q_values(online_params, next_state)
    best_action = max(range(len(actions)), key=lambda i: online_q[i])
    return reward + gamma * q_values(target_params, next_state)[best_action]

transition = (2, 1, 0.0, 3, False)
_, _, reward, next_state, done = transition
print('DQN target       ', round(dqn_target(target, reward, next_state, done), 3))
print('Double DQN target', round(double_dqn_target(online, target, reward, next_state, done), 3))

## 実務で確認する点

深層強化学習では、報酬設計、観測設計、探索率、replay buffer の分布、target network の同期周期、勾配クリップ、評価エピソードを分けて確認します。学習曲線だけでなく、Q 値の大きさ、行動分布、失敗時の軌跡を見ると、発散、過大評価、探索不足を切り分けやすくなります。

DQN の中心は「ベルマン更新をニューラルネットの教師あり回帰として解く」ことです。そこへ replay、target network、Double DQN、優先度付き replay、dueling network などの安定化を積み重ねます。